# Week 3 — Tools and the agent loop by hand

An "agent" is a loop: call the model → if it asks for a tool, run it → append the result → call again. You write that loop in about 30 lines, with four read-only tools over the procure-to-pay tables.

**Core:** steps 1–5 and the self-check. **Minimum viable week:** steps 2–4 with at least two tools.
**Stretch:** rebuild the loop with LangChain `bind_tools` and compare; delete the line that appends the tool result and read the API error.

In [ ]:
# Setup (run first)
import subprocess, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "course-tools @ git+https://github.com/YOUR-GITHUB-ORG/harness-starter@v1#subdirectory=course-tools"],
                   check=True)
    from google.colab import drive
    drive.mount("/content/drive")

import json, re, sqlite3
import pandas as pd
import anthropic
import course_tools as ct
from course_tools.meter import Meter

env = ct.bootstrap()
AGENT_MODEL = ct.MODELS["haiku"]
client = anthropic.Anthropic()
meter = Meter()

## Step 1 — Query the tables by hand first

Follow PO-1001 to its goods receipt and to any invoice already recorded. Write the SQL yourself before you ask a model to do it: the tools you build next are these queries with a name and a description.

The connection is opened **read-only** (`mode=ro`): the tools below cannot change data, whatever the model asks for.

In [ ]:
con = sqlite3.connect(f"file:{env.db}?mode=ro", uri=True)
con.row_factory = sqlite3.Row
q = lambda sql, *args: pd.read_sql_query(sql, con, params=args)

# TODO: three queries with ? placeholders: the PO lines, the goods receipt lines, and invoices for PO-1001.

## Step 2 — Four tools

Rules for every tool:

- Validate IDs with a pattern (`^S\d{2}$`, `^PO-\d{4}$`, or a GSTIN) **before** they reach SQL, and use `?` placeholders.
- Return a small dict of named fields, never whole rows you do not need.
- For an unknown ID, return an explicit error that tells the model what to do next, for example `{"error": "PO-9999 not found. Check the PO number on the invoice, or ask a person."}`. Never return an empty result that looks valid.

`get_supplier` is provided as the worked example. Write the other three.

In [ ]:
PO_RE = re.compile(r"^PO-\d{4}$")
SUPPLIER_RE = re.compile(r"^S\d{2}$")
GSTIN_RE = re.compile(r"^\d{2}[A-Z]{5}\d{4}[A-Z][1-9A-Z]Z[0-9A-Z]$")


def get_supplier(supplier_ref: str) -> dict:
    """Supplier master record, by supplier ID (S01) or GSTIN."""
    if SUPPLIER_RE.match(supplier_ref):
        row = con.execute("select * from suppliers where supplier_id = ?", (supplier_ref,)).fetchone()
    elif GSTIN_RE.match(supplier_ref):
        row = con.execute("select * from suppliers where gstin = ?", (supplier_ref,)).fetchone()
    else:
        return {"error": f"{supplier_ref!r} is not a supplier ID (like S01) or a GSTIN."}
    if row is None:
        return {"error": f"Supplier {supplier_ref} not found in the supplier master. Treat as an unknown supplier."}
    return {k: row[k] for k in ("supplier_id", "name", "gstin", "state_code", "bank_account", "ifsc")}


def get_purchase_order(po_id: str) -> dict:
    """Purchase order header and lines."""
    # TODO: validate po_id with PO_RE; query purchase_orders and po_lines (join skus for description and gst_rate)
    #       return {"po_id", "supplier_id", "po_date", "status", "lines": [...]}, or an explicit {"error": "... not found ..."}
    raise NotImplementedError


def get_goods_receipt(po_id: str) -> dict:
    """Quantities ordered and received per PO line, across all goods receipts."""
    # TODO: validate; for each PO line return line_no, sku_id, ordered, received (sum over grn_lines; 0 if none)
    #       plus "fully_received": True/False
    raise NotImplementedError


def find_similar_invoices(supplier_id: str, amount: float | None = None) -> dict:
    """Past invoices from this supplier; with an amount, only those within ₹1 of it."""
    # TODO: validate; query the invoices table for this supplier (and amount, if given)
    raise NotImplementedError


print(get_goods_receipt("PO-1001"))
print(get_purchase_order("PO-9999"))

### Tool definitions: the model's only manual

The model never sees your Python. It sees a name, a description and an input schema, re-sent on every turn. Require only the arguments a tool cannot work without: over-required fields make the model invent values.

In [ ]:
TOOLS = [
    {"name": "get_supplier", "description": "Look up a supplier in the supplier master by supplier ID (like S01) or GSTIN.",
     "input_schema": {"type": "object", "properties": {"supplier_ref": {"type": "string"}}, "required": ["supplier_ref"]}},
    # TODO: definitions for get_purchase_order, get_goods_receipt and find_similar_invoices
]
TOOL_FUNCS = {"get_supplier": get_supplier, "get_purchase_order": get_purchase_order,
              "get_goods_receipt": get_goods_receipt, "find_similar_invoices": find_similar_invoices}

## Step 3 — The loop

Write `run_agent(messages, tools, max_steps=8)`:

1. Call the model with the tools.
2. Append the assistant's reply to `messages` (the message list is the agent's whole memory).
3. If `stop_reason` is not `"tool_use"`, return the reply text.
4. Otherwise run **every** tool the model asked for and append **one** `tool_result` per tool call, with the same `tool_use_id`, all in one user message. A tool that raises becomes a result flagged `"is_error": True`: the loop never crashes because a tool did.
5. **Step 5 of the assignment:** at `max_steps`, make one last call in which the model **may not use tools** (`tool_choice={"type": "none"}`), asking for a status report. Never end a run without a report.

In [ ]:
AGENT_SYSTEM = """You are an accounts-payable assistant for Nilgiri Distributors. Use the tools to look things up;
never guess IDs or quantities. When you have the answer, reply with a short, factual summary."""

def run_agent(messages: list, tools: list, max_steps: int = 8, verbose: bool = True) -> str:
    for step in range(1, max_steps + 1):
        # TODO 1: response = client.messages.create(model=AGENT_MODEL, max_tokens=1024, system=AGENT_SYSTEM,
        #                                            tools=tools, messages=messages)
        # TODO 2: meter.record("agent", response.usage, AGENT_MODEL)
        # TODO 3: append {"role": "assistant", "content": response.content} to messages
        # TODO 4: if response.stop_reason != "tool_use": return the text of the reply
        # TODO 5: for each block with block.type == "tool_use": run TOOL_FUNCS[block.name](**block.input)
        #         inside try/except; build {"type": "tool_result", "tool_use_id": block.id,
        #         "content": <JSON string>, "is_error": <bool>}
        # TODO 6: append ONE user message whose content is the list of tool results
        raise NotImplementedError
    # TODO 7 (step 5 of the assignment): add a text block to the last user message asking for a status report,
    #         make one final call in which the model may not use tools (pass tools=tools and
    #         tool_choice={"type": "none"}: the API needs the tools because the history contains tool blocks),
    #         append it, and return its text.
    raise NotImplementedError

## Step 4 — Ask, and read the trace

*Predict* which tools the model will call, in what order, before you run it.

In [ ]:
messages = [{"role": "user", "content": "Has PO-1001 been fully received, and has it been invoiced?"}]
answer = run_agent(messages, TOOLS)
print("\n" + answer)

Print the message list. Every `tool_use` has an `id`, and the next user message carries a `tool_result` with the same id. *Stretch:* delete the line that appends the tool results and read the API's error.

In [ ]:
for m in messages:
    content = m["content"]
    if isinstance(content, str):
        print(m["role"], ":", content[:100])
    else:
        for b in content:
            b = b if isinstance(b, dict) else b.model_dump()
            print(m["role"], ":", b["type"], {k: v for k, v in b.items() if k in ("name", "input", "tool_use_id", "text")})

## Resource meter: input tokens per turn

Every turn re-sends the whole history and every tool schema. *Look at the input tokens printed at each step:* why do they grow?

In [ ]:
meter.report()

## Break it: SQL injection

This tool builds SQL with an f-string. Inject `PO-1001' OR '1'='1` and see what comes back. Then explain why your `get_purchase_order` is not vulnerable: two separate defences.

In [ ]:
def unsafe_get_po(po_id: str):
    return [dict(r) for r in con.execute(f"select * from purchase_orders where po_id = '{po_id}'").fetchall()]

print(len(unsafe_get_po("PO-1001")), "row for a normal ID")
print(len(unsafe_get_po("PO-1001' OR '1'='1")), "rows for the injected ID")
print(get_purchase_order("PO-1001' OR '1'='1"))

## Self-check

In [ ]:
ct.selfcheck(3)